# Phase 3 Query API Walkthrough

This notebook walks through the Phase 3 query API and operational analytics endpoints.
It uses the real FastAPI routes with an in-process ASGI client, and it seeds demo data through `POST /v1/runs` so the query examples exercise the same persistence path as production ingestion.


## 1. Imports And Notebook Helpers

The helpers below keep the examples compact: they create a database session, build an in-process API client, clone validated telemetry fixtures, render response tables, and remove only rows created by this notebook.


In [ ]:
from collections.abc import AsyncIterator
from contextlib import asynccontextmanager
from dataclasses import asdict, is_dataclass
from datetime import UTC, date, datetime, timedelta
from decimal import Decimal
import json
from pathlib import Path
import sys
from typing import Any

from IPython.display import Markdown, display
from httpx import ASGITransport, AsyncClient, Response
from sqlalchemy import delete
from sqlalchemy.ext.asyncio import AsyncSession, async_sessionmaker

repo_root = Path.cwd()
if not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent

src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from obs_platform.config import DatabaseOnlySettings
from obs_platform.database import create_engine, wait_for_database
from obs_platform.db.models import AgentRun, LLMCall, Span, ToolCall
from obs_platform.main import create_app
from obs_platform.routes import analytics, runs
from obs_platform.telemetry.v1 import load_fixture

DEMO_AGENT_VERSION = "notebook-phase-3"
DEMO_STARTED_AFTER = "2045-01-01T00:00:00Z"
DEMO_STARTED_BEFORE = "2045-01-31T23:59:59Z"
DEMO_RUN_IDS = [
    "notebook-phase-3-healthy",
    "notebook-phase-3-tool-error",
    "notebook-phase-3-pending",
    "notebook-phase-3-trajectory-error",
    "notebook-phase-3-policy",
    "notebook-phase-3-approved",
    "notebook-phase-3-hitl-lifecycle",
]


def jsonable(value: Any) -> Any:
    if is_dataclass(value):
        return asdict(value)
    if isinstance(value, datetime | date):
        return value.isoformat()
    if isinstance(value, Decimal):
        return float(value)
    return value


def table(rows: list[dict[str, Any]]) -> None:
    if not rows:
        display(Markdown("_No rows._"))
        return

    columns = list(rows[0].keys())
    header = "| " + " | ".join(columns) + " |"
    divider = "| " + " | ".join("---" for _ in columns) + " |"
    body = []
    for row in rows:
        values = []
        for column in columns:
            value = jsonable(row[column])
            if isinstance(value, dict | list):
                value = json.dumps(value, sort_keys=True)
            values.append(str(value).replace("\n", " ").replace("|", "\\|"))
        body.append("| " + " | ".join(values) + " |")
    display(Markdown("\n".join([header, divider, *body])))


def compact_json(value: Any, *, max_chars: int = 5000) -> None:
    text = json.dumps(value, indent=2, sort_keys=True, default=jsonable)
    if len(text) > max_chars:
        print(text[:max_chars])
        print("\n... JSON truncated for display ...")
    else:
        print(text)


def select_fields(items: list[dict[str, Any]], fields: list[str]) -> list[dict[str, Any]]:
    return [{field: item.get(field) for field in fields} for item in items]


def contains_exact_key(value: Any, key: str) -> bool:
    if isinstance(value, dict):
        return key in value or any(contains_exact_key(child, key) for child in value.values())
    if isinstance(value, list):
        return any(contains_exact_key(item, key) for item in value)
    return False


def clone_fixture(
    name: str,
    run_id: str,
    *,
    started_at: datetime,
    scenario_id: str,
    model: str,
    provider: str = "notebook-provider",
):
    event = load_fixture(name)
    event.run_id = run_id
    event.started_at = started_at
    if event.execution_latency_ms is not None:
        event.completed_at = started_at + timedelta(milliseconds=event.execution_latency_ms)
    event.scenario_id = scenario_id
    event.agent_version = DEMO_AGENT_VERSION
    for llm_call in event.llm_calls:
        llm_call.model = model
        llm_call.provider = provider
    return event


async def delete_runs(session: AsyncSession, run_ids: list[str]) -> None:
    for model in (LLMCall, ToolCall, Span, AgentRun):
        await session.execute(delete(model).where(model.run_id.in_(run_ids)))
    await session.commit()


@asynccontextmanager
async def api_client(session: AsyncSession) -> AsyncIterator[AsyncClient]:
    async def notebook_session() -> AsyncIterator[AsyncSession]:
        yield session

    app = create_app()
    app.dependency_overrides[runs.get_session] = notebook_session
    app.dependency_overrides[analytics.get_session] = notebook_session
    async with AsyncClient(
        transport=ASGITransport(app=app),
        base_url="http://testserver",
    ) as client:
        yield client


async def api_request(
    session: AsyncSession,
    method: str,
    path: str,
    **kwargs: Any,
) -> Response:
    async with api_client(session) as client:
        return await client.request(method, path, **kwargs)


settings = DatabaseOnlySettings()
engine = create_engine(settings.db)
await wait_for_database(engine)
Session = async_sessionmaker(engine, expire_on_commit=False)

display(Markdown(f"Connected to PostgreSQL at `{settings.db.host}:{settings.db.port}/{settings.db.name}`."))


## 2. Seed Demo Runs Through The API

Each row below is posted to `POST /v1/runs`. The seeded runs use a far-future January 2045 time window so analytics examples can be scoped without mixing in unrelated local data.


In [ ]:
base_started_at = datetime(2045, 1, 10, 12, 0, tzinfo=UTC)
seed_events = [
    clone_fixture(
        "healthy_success",
        "notebook-phase-3-healthy",
        started_at=base_started_at,
        scenario_id="notebook-scenario-grid-repair",
        model="notebook-fast-model",
    ),
    clone_fixture(
        "tool_failure",
        "notebook-phase-3-tool-error",
        started_at=base_started_at + timedelta(minutes=10),
        scenario_id="notebook-scenario-tool-failure",
        model="notebook-fast-model",
    ),
    clone_fixture(
        "hitl_pending",
        "notebook-phase-3-pending",
        started_at=base_started_at + timedelta(minutes=20),
        scenario_id="notebook-scenario-hitl",
        model="notebook-review-model",
    ),
    clone_fixture(
        "trajectory_error",
        "notebook-phase-3-trajectory-error",
        started_at=base_started_at + timedelta(minutes=30),
        scenario_id="notebook-scenario-trajectory",
        model="notebook-review-model",
    ),
    clone_fixture(
        "policy_violation",
        "notebook-phase-3-policy",
        started_at=base_started_at + timedelta(minutes=40),
        scenario_id="notebook-scenario-policy",
        model="notebook-premium-model",
    ),
    clone_fixture(
        "hitl_approved",
        "notebook-phase-3-approved",
        started_at=base_started_at + timedelta(minutes=50),
        scenario_id="notebook-scenario-hitl",
        model="notebook-premium-model",
    ),
]

async with Session() as session:
    await delete_runs(session, DEMO_RUN_IDS)
    responses = []
    async with api_client(session) as client:
        for event in seed_events:
            response = await client.post("/v1/runs", json=event.model_dump(mode="json"))
            responses.append(
                {
                    "fixture_run_id": event.run_id,
                    "http_status": response.status_code,
                    "response_body": response.json(),
                }
            )
    await session.commit()

table(responses)


## 3. `GET /v1/runs`: Find Runs

The list endpoint returns a paginated envelope with lightweight `RunSummary` items. It is meant for scanning and filtering, so it intentionally omits spans, tool calls, LLM payloads, and full trajectory details.


In [ ]:
async with Session() as session:
    response = await api_request(
        session,
        "GET",
        "/v1/runs",
        params={"agent_version": DEMO_AGENT_VERSION, "limit": 10},
    )
    body = response.json()

print("HTTP", response.status_code)
table([
    {
        "total": body["total"],
        "limit": body["limit"],
        "offset": body["offset"],
        "items_returned": len(body["items"]),
    }
])
table(select_fields(body["items"], [
    "run_id",
    "scenario_id",
    "status",
    "event_type",
    "hitl_state",
    "started_at",
    "usage_total_tokens",
    "usage_total_estimated_cost_usd",
]))


### List Filters

Phase 3 supports exact filters for `status`, `scenario_id`, `agent_version`, and `model`, plus a `started_at` time window. Filters combine with AND semantics.


In [ ]:
filter_examples = [
    ("status", {"status": "tool_error", "agent_version": DEMO_AGENT_VERSION}),
    ("scenario_id", {"scenario_id": "notebook-scenario-hitl", "agent_version": DEMO_AGENT_VERSION}),
    ("model", {"model": "notebook-premium-model", "agent_version": DEMO_AGENT_VERSION}),
    (
        "time window",
        {
            "started_after": "2045-01-10T12:15:00Z",
            "started_before": "2045-01-10T12:45:00Z",
            "agent_version": DEMO_AGENT_VERSION,
        },
    ),
    (
        "AND semantics",
        {
            "scenario_id": "notebook-scenario-hitl",
            "model": "notebook-premium-model",
            "agent_version": DEMO_AGENT_VERSION,
        },
    ),
]

rows = []
async with Session() as session:
    for label, params in filter_examples:
        response = await api_request(session, "GET", "/v1/runs", params=params)
        body = response.json()
        rows.append(
            {
                "example": label,
                "http_status": response.status_code,
                "total": body["total"],
                "run_ids": [item["run_id"] for item in body["items"]],
            }
        )

table(rows)


### Validation Example

The API uses Pydantic/FastAPI validation for query parameters. A `limit` above the configured maximum is rejected instead of silently clamped.


In [ ]:
async with Session() as session:
    response = await api_request(session, "GET", "/v1/runs", params={"limit": 101})

print("HTTP", response.status_code)
compact_json(response.json(), max_chars=1600)


## 4. `GET /v1/runs/{run_id}`: Inspect One Run

The detail endpoint returns the full persisted view of a run: request/final output, flat ordered spans, ordered tool calls, ordered LLM calls, HITL state, usage totals, and runtime error information.


In [ ]:
DETAIL_RUN_ID = "notebook-phase-3-tool-error"

async with Session() as session:
    response = await api_request(session, "GET", f"/v1/runs/{DETAIL_RUN_ID}")
    detail = response.json()

print("HTTP", response.status_code)
table([
    {
        "run_id": detail["run_id"],
        "status": detail["status"],
        "event_type": detail["event_type"],
        "scenario_id": detail["scenario_id"],
        "spans": len(detail["spans"]),
        "tool_calls": len(detail["tool_calls"]),
        "llm_calls": len(detail["llm_calls"]),
        "has_runtime_error": detail["runtime_error"] is not None,
    }
])

display(Markdown("### usage"))
compact_json(detail["usage"], max_chars=1200)

display(Markdown("### runtime_error"))
compact_json(detail["runtime_error"], max_chars=1200)


### Flat Ordered Trajectory

Spans, tool calls, and LLM calls are returned as separate flat lists. `span_id`, `tool_call_id`, and `llm_call_id` are external IDs; internal database surrogate IDs are not exposed.


In [ ]:
display(Markdown("### spans"))
table(select_fields(detail["spans"], [
    "sequence",
    "span_id",
    "parent_span_id",
    "name",
    "status",
]))

display(Markdown("### tool_calls"))
table(select_fields(detail["tool_calls"], [
    "sequence",
    "tool_call_id",
    "span_id",
    "tool_name",
    "status",
    "latency_ms",
    "retry_count",
]))

display(Markdown("### llm_calls"))
table(select_fields(detail["llm_calls"], [
    "sequence",
    "llm_call_id",
    "span_id",
    "call_type",
    "provider",
    "model",
    "status",
    "total_tokens",
    "estimated_cost_usd",
]))

table([
    {"check": "spans are sequence ordered", "value": [item["sequence"] for item in detail["spans"]] == sorted(item["sequence"] for item in detail["spans"] )},
    {"check": "tool calls are sequence ordered", "value": [item["sequence"] for item in detail["tool_calls"]] == sorted(item["sequence"] for item in detail["tool_calls"] )},
    {"check": "no exact internal `id` key exposed", "value": not contains_exact_key(detail, "id")},
])


### Missing Run

Unknown run IDs return `404`, not an empty successful response.


In [ ]:
async with Session() as session:
    response = await api_request(session, "GET", "/v1/runs/notebook-phase-3-missing")

print("HTTP", response.status_code)
compact_json(response.json(), max_chars=1200)


## 5. HITL Pending To Approved Lifecycle

This example posts a pending HITL snapshot, fetches the detail response, then posts the approved final snapshot for the same `run_id` and fetches detail again. The row evolves by upsert: carried-over trajectory IDs remain stable and the post-approval tool call is appended.


In [ ]:
LIFECYCLE_RUN_ID = "notebook-phase-3-hitl-lifecycle"
pending = clone_fixture(
    "hitl_pending",
    LIFECYCLE_RUN_ID,
    started_at=datetime(2045, 1, 20, 12, 0, tzinfo=UTC),
    scenario_id="notebook-scenario-hitl-lifecycle",
    model="notebook-review-model",
)
approved = clone_fixture(
    "hitl_approved",
    LIFECYCLE_RUN_ID,
    started_at=datetime(2045, 1, 20, 12, 0, tzinfo=UTC),
    scenario_id="notebook-scenario-hitl-lifecycle",
    model="notebook-review-model",
)

async with Session() as session:
    await delete_runs(session, [LIFECYCLE_RUN_ID])
    async with api_client(session) as client:
        pending_post = await client.post("/v1/runs", json=pending.model_dump(mode="json"))
        pending_detail = (await client.get(f"/v1/runs/{LIFECYCLE_RUN_ID}")).json()
        await session.commit()

        approved_post = await client.post("/v1/runs", json=approved.model_dump(mode="json"))
        approved_detail = (await client.get(f"/v1/runs/{LIFECYCLE_RUN_ID}")).json()
    await session.commit()

pending_span_ids = [item["span_id"] for item in pending_detail["spans"]]
approved_span_ids = [item["span_id"] for item in approved_detail["spans"]]
pending_tool_ids = [item["tool_call_id"] for item in pending_detail["tool_calls"]]
approved_tool_ids = [item["tool_call_id"] for item in approved_detail["tool_calls"]]

summary_rows = [
    {
        "snapshot": "pending",
        "post_status": pending_post.status_code,
        "event_type": pending_detail["event_type"],
        "status": pending_detail["status"],
        "hitl_state": pending_detail["hitl"]["state"],
        "spans": len(pending_detail["spans"]),
        "tool_calls": len(pending_detail["tool_calls"]),
        "final_result": pending_detail["final_result"] is not None,
    },
    {
        "snapshot": "approved",
        "post_status": approved_post.status_code,
        "event_type": approved_detail["event_type"],
        "status": approved_detail["status"],
        "hitl_state": approved_detail["hitl"]["state"],
        "spans": len(approved_detail["spans"]),
        "tool_calls": len(approved_detail["tool_calls"]),
        "final_result": approved_detail["final_result"] is not None,
    },
]

table(summary_rows)

display(Markdown("### HITL blocks"))
compact_json({"pending": pending_detail["hitl"], "approved": approved_detail["hitl"]}, max_chars=2500)

display(Markdown("### lifecycle checks"))
table([
    {"check": "pending spans carried into approved detail", "value": set(pending_span_ids).issubset(set(approved_span_ids))},
    {"check": "pending tool calls carried into approved detail", "value": set(pending_tool_ids).issubset(set(approved_tool_ids))},
    {"check": "new post-approval tool call IDs", "value": sorted(set(approved_tool_ids) - set(pending_tool_ids))},
])


## 6. `GET /v1/analytics/overview`: Runtime Summary

The overview endpoint answers the system-level question: how many runs are in scope, how often terminal runs succeeded, how much latency was observed, and how much aggregate token/cost usage was recorded. Pending HITL runs are counted in `run_counts`, but excluded from the runtime success-rate denominator.


In [ ]:
async with Session() as session:
    response = await api_request(
        session,
        "GET",
        "/v1/analytics/overview",
        params={
            "started_after": DEMO_STARTED_AFTER,
            "started_before": DEMO_STARTED_BEFORE,
        },
    )
    overview = response.json()

print("HTTP", response.status_code)
compact_json(overview, max_chars=3000)

table([
    {
        "metric": "runtime_success_rate",
        "value": overview["runtime_success_rate"],
        "note": "terminal run_final rows only",
    },
    {
        "metric": "awaiting_approval_count",
        "value": overview["run_counts"]["by_status"]["awaiting_approval"],
        "note": "included in counts, excluded from success-rate denominator",
    },
])


## 7. `GET /v1/analytics/tools`: Tool Behavior

Tool analytics groups by free-text `tool_name`, returns raw status counts, computes `(failure + error) / total` as `failure_rate`, and sorts by call volume descending.


In [ ]:
async with Session() as session:
    response = await api_request(
        session,
        "GET",
        "/v1/analytics/tools",
        params={
            "started_after": DEMO_STARTED_AFTER,
            "started_before": DEMO_STARTED_BEFORE,
        },
    )
    tools_body = response.json()

print("HTTP", response.status_code)
table(tools_body["items"])


## 8. `GET /v1/analytics/usage`: Agent LLM Usage

Usage analytics reads only `llm_calls`: total agent-side token/cost usage, plus dynamic breakdowns by `(provider, model)` and by `call_type`. It does not include future evaluator or judge LLM usage.


In [ ]:
async with Session() as session:
    response = await api_request(
        session,
        "GET",
        "/v1/analytics/usage",
        params={
            "started_after": DEMO_STARTED_AFTER,
            "started_before": DEMO_STARTED_BEFORE,
        },
    )
    usage_body = response.json()

print("HTTP", response.status_code)

display(Markdown("### total"))
compact_json(usage_body["total"], max_chars=1200)

display(Markdown("### by_model"))
table(usage_body["by_model"])

display(Markdown("### by_call_type"))
table(usage_body["by_call_type"])


## 9. Endpoint Map

- `GET /v1/runs`: find and filter runs with lightweight summaries.
- `GET /v1/runs/{run_id}`: inspect one complete run trajectory and state.
- `GET /v1/analytics/overview`: summarize runtime outcomes, latency, usage, and counts.
- `GET /v1/analytics/tools`: compare tool volume, failures, and latency.
- `GET /v1/analytics/usage`: account for agent-side LLM tokens and estimated cost.


## 10. Cleanup

This removes only the deterministic demo run IDs created by the notebook.


In [ ]:
async with Session() as session:
    await delete_runs(session, DEMO_RUN_IDS)

await engine.dispose()
display(Markdown("Notebook demo rows removed and engine disposed."))
